# C3R: High-Iteration COCO128 Research Notebook

Run **all five Python cells in order**. At Step 3, upload the included `03A_UPLOAD_THIS_Concept_Candidates.py`. The detector remains **frozen**. Increased computation refers to spatial controls and validation, not training epochs. See `README_ROMAN_URDU.md` in the ZIP.

In [ ]:

# C3R — STEP 1: REAL COCO OBJECT DETECTION

import torch
import torchvision

# TorchVision compatibility
if not hasattr(torchvision, "_is_tracing"):
    torchvision._is_tracing = torch.jit.is_tracing

from pathlib import Path
import zipfile
import requests
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from torchvision.transforms.functional import to_tensor
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights
)

# 1. Download real MS COCO images
root = Path(globals().get("C3R_OUTPUT_DIR", "/content/c3r_step1"))
images_dir = root / "coco128/images/train2017"

if not images_dir.exists() or not list(images_dir.glob("*.jpg")):
    print("Downloading COCO128...")
    root.mkdir(parents=True, exist_ok=True)

    url = (
        "https://github.com/ultralytics/assets/"
        "releases/download/v0.0.0/coco128.zip"
    )

    archive = root / "coco128.zip"

    with requests.get(url, stream=True, timeout=90) as response:
        response.raise_for_status()
        with open(archive, "wb") as f:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    with zipfile.ZipFile(archive) as z:
        z.extractall(root)

image_files = sorted(images_dir.glob("*.jpg"))
assert image_files, "COCO128 images not found."

preferred = images_dir / "000000000009.jpg"
image_path = preferred if preferred.exists() else image_files[0]
image = Image.open(image_path).convert("RGB")

print("Selected COCO image:", image_path.name)

# 2. Load pretrained frozen Faster R-CNN
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn(
    weights=weights,
    min_size=480,
    max_size=720
)

model = model.to(device).eval()

for param in model.parameters():
    param.requires_grad_(False)

print("Model: Faster R-CNN ResNet50 FPN")
print("Device:", device)
print("Detector: FROZEN")

# 3. Run real inference
with torch.inference_mode():
    predictions = model([
        to_tensor(image).to(device)
    ])[0]

categories = weights.meta["categories"]

# 4. Visualize detected objects
fig, ax = plt.subplots(figsize=(11, 7))
ax.imshow(image)

detected_count = 0

for box, score, label in zip(
    predictions["boxes"].cpu(),
    predictions["scores"].cpu(),
    predictions["labels"].cpu()
):
    if float(score) < 0.50 or detected_count >= 8:
        break

    x1, y1, x2, y2 = box.tolist()
    class_name = categories[int(label)]

    ax.add_patch(
        Rectangle(
            (x1, y1),
            x2-x1,
            y2-y1,
            fill=False,
            edgecolor="lime",
            linewidth=2
        )
    )

    ax.text(
        x1,
        max(y1-4, 0),
        f"{class_name}: {float(score):.3f}",
        color="white",
        fontsize=10,
        bbox=dict(facecolor="green", alpha=0.8, pad=2)
    )

    detected_count += 1

ax.set_title(
    "C3R Step 1 — Frozen Faster R-CNN / MS COCO",
    fontsize=14
)
ax.axis("off")
plt.tight_layout()

output_path = root / "step1_clean_detection.png"
plt.savefig(output_path, dpi=170, bbox_inches="tight")
plt.show()

print("Detected objects:", detected_count)
print("Saved image:", output_path)
print("STEP 1 COMPLETED")


In [ ]:

# C3R — STEP 2: OBJECT-SPECIFIC RoI FEATURES

import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from pathlib import Path
from torchvision.transforms.functional import to_tensor
from matplotlib.patches import Rectangle

model.eval()
for p in model.parameters():
    p.requires_grad_(False)

# Select top 2 confidently detected objects
selected = torch.where(
    predictions["scores"] >= 0.50
)[0][:2]

if len(selected) == 0:
    raise ValueError("No detections above threshold.")

boxes = predictions["boxes"][selected].detach()
labels = predictions["labels"][selected].detach()
scores = predictions["scores"][selected].detach()

# Extract backbone and aligned RoI features
with torch.no_grad():
    input_tensor = to_tensor(image).to(device)
    images_t, _ = model.transform([input_tensor], None)

    backbone_features = model.backbone(images_t.tensors)

    old_w, old_h = image.size
    new_h, new_w = images_t.image_sizes[0]

    scale = torch.tensor(
        [new_w / old_w, new_h / old_h,
         new_w / old_w, new_h / old_h],
        device=device,
        dtype=boxes.dtype
    )

    transformed_boxes = boxes * scale

    roi_pooled = model.roi_heads.box_roi_pool(
        backbone_features,
        [transformed_boxes],
        images_t.image_sizes
    )

    roi_embeddings = model.roi_heads.box_head(
        roi_pooled
    )

print("Detected objects:", len(selected))
print("RoI pooled shape:", tuple(roi_pooled.shape))
print("RoI embedding shape:", tuple(roi_embeddings.shape))

# Generate object-specific class-conditioned maps
original = np.asarray(image)
H, W = original.shape[:2]

fig, axes = plt.subplots(
    1, len(selected) + 1,
    figsize=(6 * (len(selected) + 1), 5)
)

axes[0].imshow(original)
axes[0].set_title("Clean COCO Detection")
axes[0].axis("off")

for i in range(len(selected)):
    class_id = int(labels[i].item())
    name = categories[class_id]
    confidence = float(scores[i].item())

    # Local gradient at the pooled RoI representation
    roi = (
        roi_pooled[i:i+1]
        .detach()
        .clone()
        .requires_grad_(True)
    )

    with torch.enable_grad():
        emb = model.roi_heads.box_head(roi)

        logits, _ = model.roi_heads.box_predictor(emb)

        target_logit = logits[0, class_id]

        grads = torch.autograd.grad(
            target_logit, roi
        )[0]

        weights_cam = grads.mean(
            dim=(2, 3), keepdim=True
        )

        cam = F.relu(
            (weights_cam * roi).sum(dim=1)[0]
        )

    cam = cam.detach()
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)

    x1, y1, x2, y2 = boxes[i].cpu().tolist()

    x1 = max(0, min(W-1, int(x1)))
    y1 = max(0, min(H-1, int(y1)))
    x2 = max(x1+1, min(W, int(x2)))
    y2 = max(y1+1, min(H, int(y2)))

    patch_h = y2 - y1
    patch_w = x2 - x1

    resized_cam = F.interpolate(
        cam[None, None],
        size=(patch_h, patch_w),
        mode="bilinear",
        align_corners=False
    )[0, 0].cpu().numpy()

    heatmap = np.zeros((H, W), dtype=np.float32)
    heatmap[y1:y2, x1:x2] = resized_cam

    ax = axes[i+1]
    ax.imshow(original)

    masked_heat = np.ma.masked_where(
        heatmap <= 0, heatmap
    )

    ax.imshow(
        masked_heat,
        cmap="jet",
        alpha=0.55,
        vmin=0,
        vmax=1
    )

    ax.add_patch(
        Rectangle(
            (x1, y1),
            patch_w,
            patch_h,
            fill=False,
            edgecolor="lime",
            linewidth=2
        )
    )

    ax.set_title(
        f"{name} ({confidence:.3f})\n"
        "RoI class-response map"
    )
    ax.axis("off")

    print(
        f"Object {i+1}: {name}, "
        f"confidence={confidence:.4f}"
    )

plt.suptitle(
    "C3R Step 2 — Object-Level RoI Attribution",
    fontsize=15
)
plt.tight_layout()

output_dir = Path(globals().get("C3R_OUTPUT_DIR", "/content/c3r_step1"))
output_dir.mkdir(parents=True, exist_ok=True)

figure_path = output_dir / "step2_roi_response.png"
plt.savefig(
    figure_path, dpi=200,
    bbox_inches="tight"
)
plt.show()

# Save object representations
feature_path = output_dir / "step2_roi_features.npz"

np.savez_compressed(
    feature_path,
    features=roi_embeddings.detach().cpu().numpy(),
    boxes=boxes.cpu().numpy(),
    labels=labels.cpu().numpy(),
    scores=scores.cpu().numpy()
)

print("\nSTEP 2 COMPLETED")
print("Figure:", figure_path)
print("Features:", feature_path)


In [ ]:

# STEP 3 — Run concept-candidate analysis

from google.colab import files

uploaded = files.upload()

script = next(
    (name for name in uploaded if name.endswith(".py")),
    None
)

if script is None:
    raise ValueError("Upload the Step 3 Python file.")

with open(script, "r", encoding="utf-8") as f:
    code = f.read()

exec(compile(code, script, "exec"), globals())


In [ ]:
# C3R STEP 3B — Higher-replication, foreground-restricted region validation.
# Run in the SAME Colab runtime after the uploaded Step 3A file.
# The 120 spatial controls/region (up from 3 in original) are NOT independent samples.

import hashlib
import json
import time
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image as PILImage, ImageFilter

required_3b = ('source', 'image', 'segments', 'x1', 'y1', 'x2', 'y2',
               'record', 'fixed_box_class_logits', 'predicted_class',
               'alternative_class', 'base_margin', 'categories')
missing_3b = [k for k in required_3b if k not in globals()]
if missing_3b:
    raise RuntimeError('Run Step 3A first; missing: ' + ', '.join(missing_3b))

N_CONTROLS_3B = int(globals().get('N_CONTROLS_3B', 120))
MAX_REGIONS_3B = int(globals().get('MAX_REGIONS_3B', 5))
BATCH_3B = int(globals().get('BATCH_3B', 2))
SEED_3B = 42
if N_CONTROLS_3B < 3 or BATCH_3B < 1:
    raise ValueError('N_CONTROLS_3B must be >= 3 and BATCH_3B >= 1.')
source = np.asarray(source, dtype=np.uint8)
H, W = source.shape[:2]
ch, cw = y2-y1, x2-x1
crop = source[y1:y2, x1:x2].copy()

# Estimate object foreground in the local, padded context.
pad = max(12, int(0.30 * max(ch, cw)))
px1, py1 = max(0, x1-pad), max(0, y1-pad)
px2, py2 = min(W, x2+pad), min(H, y2+pad)
context = source[py1:py2, px1:px2].copy()
rx, ry = x1-px1, y1-py1
mask_method = 'GrabCut foreground estimate (not ground truth)'
try:
    cv2.setRNGSeed(SEED_3B)
    gc_mask = np.zeros(context.shape[:2], dtype=np.uint8)
    bg_model, fg_model = np.zeros((1, 65), np.float64), np.zeros((1, 65), np.float64)
    cv2.grabCut(cv2.cvtColor(context, cv2.COLOR_RGB2BGR), gc_mask,
                (rx, ry, cw, ch), bg_model, fg_model, 5, cv2.GC_INIT_WITH_RECT)
    foreground = np.isin(gc_mask[ry:ry+ch, rx:rx+cw], [cv2.GC_FGD, cv2.GC_PR_FGD])
    foreground = cv2.morphologyEx(foreground.astype(np.uint8), cv2.MORPH_CLOSE,
                                  np.ones((3, 3), np.uint8)).astype(bool)
    if not 0.06 <= foreground.mean() <= 0.94:
        raise ValueError('Uninformative foreground mask')
except (cv2.error, ValueError) as e:
    foreground = np.ones((ch, cw), dtype=bool)
    mask_method = 'Bbox-only fallback (not object mask)'
    print('[Step 3B] WARNING:', str(e))

selected_3b = []
for item in record:
    rid = int(item['region'])
    region_mask = (segments == rid) & foreground
    n = int(region_mask.sum())
    if n >= max(25, int(0.015 * foreground.sum())):
        selected_3b.append((rid, region_mask, n))
    if len(selected_3b) >= MAX_REGIONS_3B:
        break
if not selected_3b:
    raise RuntimeError('No suitable foreground candidates. Inspect mask/segments.')
print('[Step 3B] Foreground:', mask_method)
print('[Step 3B] Selected:', [(r, n) for r, _, n in selected_3b])

rng_3b = np.random.default_rng(SEED_3B)
fg_coords_3b = np.column_stack(np.where(foreground)).astype(np.int32)
blurred = np.asarray(image.filter(ImageFilter.GaussianBlur(radius=14)).convert('RGB'),
                     dtype=np.uint8)[y1:y2, x1:x2]
neutral_color = np.median(crop[foreground], axis=0).astype(np.uint8)
METHODS_3B = ('blur', 'neutral_fill')

def random_control_3b(n, target_mask):
    best, best_overlap = None, 2.0
    for _ in range(40):
        yy, xx = fg_coords_3b[int(rng_3b.integers(len(fg_coords_3b)))].tolist()
        dist = ((fg_coords_3b[:, 0].astype(float)-yy)**2
                + (fg_coords_3b[:, 1].astype(float)-xx)**2)
        indices = np.argpartition(dist, n-1)[:n]
        m = np.zeros((ch, cw), dtype=bool)
        m[fg_coords_3b[indices, 0], fg_coords_3b[indices, 1]] = True
        overlap = float((m & target_mask).sum()/n)
        if overlap < best_overlap:
            best, best_overlap = m, overlap
        if overlap <= .25:
            break
    return best, best_overlap


def edited_image_3b(mask, method):
    edited = source.copy()
    view = edited[y1:y2, x1:x2]
    if method == 'blur':
        view[mask] = blurred[mask]
    elif method == 'neutral_fill':
        view[mask] = neutral_color
    else:
        raise ValueError(method)
    return edited


def logits_batch_3b(arrays):
    if callable(globals().get('fixed_roi_logits_batch_3a')):
        return fixed_roi_logits_batch_3a(arrays)
    return np.stack([fixed_box_class_logits(PILImage.fromarray(a)).detach().float().cpu().numpy()
                     for a in arrays])

# Compose a deterministic list so an interrupted Colab session can resume scoring.
jobs_3b, overlap_3b = [], {}
for rid, m, n in selected_3b:
    for method in METHODS_3B:
        jobs_3b.append(dict(region=rid, mask=m, method=method, kind='observed', trial=-1))
    controls = []
    for trial in range(N_CONTROLS_3B):
        cm, overlap = random_control_3b(n, m)
        controls.append(overlap)
        for method in METHODS_3B:
            jobs_3b.append(dict(region=rid, mask=cm, method=method,
                                kind='control', trial=trial))
    overlap_3b[rid] = controls

out_dir = Path(globals().get('C3R_OUTPUT_DIR', '/content/c3r_step1'))
out_dir.mkdir(parents=True, exist_ok=True)
run_hash_3b = hashlib.sha256(
    source.tobytes() + segments.astype(np.int32).tobytes() + foreground.tobytes()
    + np.asarray(globals().get('base_logits', []), dtype=np.float32).tobytes()
    + json.dumps({'n_controls': N_CONTROLS_3B, 'regions': [int(r) for r,_,_ in selected_3b],
                  'classes': [predicted_class, alternative_class], 'seed': SEED_3B}, sort_keys=True).encode()
).hexdigest()[:16]
progress_path_3b = out_dir/f'step3b_progress_{run_hash_3b}.jsonl'
completed_3b = {}
if progress_path_3b.exists():
    for line in progress_path_3b.read_text().splitlines():
        try:
            entry = json.loads(line)
            completed_3b[int(entry['job_id'])] = entry
        except (json.JSONDecodeError, ValueError, KeyError):
            pass
print(f'[Step 3B] {len(jobs_3b)} edits, {len(completed_3b)} cached; {N_CONTROLS_3B} controls per region')
start_time_3b = time.time()
with progress_path_3b.open('a', encoding='utf-8') as progress_file:
    for k in range(0, len(jobs_3b), BATCH_3B):
        pending = [(i, jobs_3b[i]) for i in range(k, min(k+BATCH_3B, len(jobs_3b)))
                   if i not in completed_3b]
        if pending:
            scores = logits_batch_3b([edited_image_3b(j['mask'], j['method']) for _, j in pending])
            for (index, job), logits in zip(pending, scores):
                margin = float(logits[predicted_class] - logits[alternative_class])
                entry = dict(job_id=index, region=int(job['region']),
                             method=job['method'], kind=job['kind'], trial=int(job['trial']),
                             edited_margin=margin, delta=float(base_margin-margin))
                completed_3b[index] = entry
                progress_file.write(json.dumps(entry)+'\n')
            progress_file.flush()
        if (k//BATCH_3B + 1) % 25 == 0 or k+BATCH_3B >= len(jobs_3b):
            print(f'[Step 3B] {min(k+BATCH_3B,len(jobs_3b))}/{len(jobs_3b)} jobs checked'
                  f' ({time.time()-start_time_3b:.0f}s elapsed)')

records_3b = [completed_3b[i] for i in range(len(jobs_3b))]
results_3b = []
for rid, region_mask, n in selected_3b:
    measurements = {}
    for method in METHODS_3B:
        rows = [r for r in records_3b if r['region']==rid and r['method']==method]
        observed = next(r['delta'] for r in rows if r['kind']=='observed')
        controls = np.asarray([r['delta'] for r in rows if r['kind']=='control'], dtype=float)
        abs_controls = np.abs(controls)
        measurements[method] = {
            'effect': float(observed), 'control_values': controls.tolist(),
            'control_abs_max': float(abs_controls.max()),
            'q95_control_abs': float(np.quantile(abs_controls, .95)),
            'outside_control_q95': bool(abs(observed) > np.quantile(abs_controls, .95)),
            'outside_sampled_control_range': bool(abs(observed) > abs_controls.max()),
            # Descriptive Monte Carlo exceedance; spatial controls are correlated.
            'absolute_control_exceedance_fraction': float((1+(abs_controls >= abs(observed)).sum())/(len(controls)+1)),
        }
    results_3b.append({
        'region': rid, 'pixels': n, 'median_control_overlap': float(np.median(overlap_3b[rid])),
        'measurements': measurements,
    })
    print('[Step 3B] Region', rid,
          '| blur', round(measurements['blur']['effect'], 3),
          '| fill', round(measurements['neutral_fill']['effect'], 3),
          '| median overlap', round(float(np.median(overlap_3b[rid])), 3))

fig, axs = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
axs[0, 0].imshow(source)
axs[0, 0].add_patch(Rectangle((x1, y1), cw, ch, fill=False, edgecolor='lime', linewidth=2))
axs[0, 0].set_title('A. Clean image / fixed target box')
axs[0, 1].imshow(crop)
axs[0, 1].imshow(np.ma.masked_where(~foreground, foreground), cmap='Greens', alpha=.45)
axs[0, 1].set_title('B. Foreground proxy (not ground truth)')
colored = crop.copy()
for idx, (rid, m, _) in enumerate(selected_3b):
    color = np.array(plt.get_cmap('tab10')(idx)[:3]) * 255
    colored[m] = np.asarray(.5*colored[m] + .5*color, dtype=np.uint8)
axs[0, 2].imshow(colored)
axs[0, 2].set_title(f'C. {len(selected_3b)} candidate regions')
for ax, (rid, m, n) in zip(axs[1, :2], selected_3b):
    ax.imshow(crop)
    ax.imshow(np.ma.masked_where(~m, m), cmap='autumn', alpha=.50)
    ax.set_title(f'Region {rid} ({n} pixels)')
if len(selected_3b) == 1:
    axs[1, 1].imshow(crop)
    axs[1, 1].set_title('D/E. One region qualified')

x_positions = np.arange(len(results_3b))
bar_width = .36
for j, method in enumerate(METHODS_3B):
    signed = [r['measurements'][method]['effect'] for r in results_3b]
    quantile = [r['measurements'][method]['q95_control_abs'] for r in results_3b]
    centers = x_positions + (j-.5)*bar_width
    axs[1, 2].bar(centers, signed, bar_width, label=method)
    axs[1, 2].scatter(centers, quantile, marker='^', color='black', s=25)
    axs[1, 2].scatter(centers, [-v for v in quantile], marker='v', color='black', s=25)
axs[1, 2].axhline(0, color='black', linewidth=.8)
axs[1, 2].set_xticks(x_positions, [f"R{r['region']}" for r in results_3b])
axs[1, 2].set_ylabel('Change in contrastive logit margin')
axs[1, 2].set_title('F. Effects vs |control| 95th percentile')
axs[1, 2].legend(fontsize=8)
for ax in axs.flat:
    if ax is not axs[1, 2]:
        ax.axis('off')
fig.suptitle(f'C3R Step 3B | {categories[predicted_class]} vs {categories[alternative_class]}\n'
             'Expanded descriptive controls; no semantic or causal validation', fontsize=13)
figure_path = out_dir/'step3b_results.png'
json_path = out_dir/'step3b_results.json'
fig.savefig(figure_path, dpi=190, bbox_inches='tight')
plt.show()
with open(json_path, 'w', encoding='utf-8') as output:
    json.dump({
        'target_class': str(categories[predicted_class]),
        'alternative_class': str(categories[alternative_class]),
        'baseline_margin': float(base_margin),
        'foreground_method': mask_method,
        'n_controls': N_CONTROLS_3B,
        'n_regions': len(selected_3b),
        'regions': results_3b,
        'calibrated_reliability': None,
        'semantic_roles_validated': False,
        'cautions': ['Spatial controls are correlated; descriptive threshold, not statistical significance',
                     'More control samples improve stability of quantile estimates but do not train the detector'],
    }, output, indent=2)
print('[Step 3B] Figure:', figure_path)
print('[Step 3B] Data:', json_path)
print('[Step 3B] Resume checkpoint:', progress_path_3b)
print('STEP 3B COMPLETED — now run Step 3C.')


In [ ]:
# C3R STEP 3C -- controlled foreground-restricted intervention screening
# Paste into ONE NEW CELL after 3B. Uses the SAME frozen Faster R-CNN.
# IMPORTANT: this does NOT establish semantic concepts, causal identification, or calibrated reliability.

import gc, json, time, hashlib, csv
from pathlib import Path
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image as PILImage, ImageFilter
from torchvision.transforms.functional import to_tensor

required_3c = ['model','device','image','source','segments','foreground','record',
               'x1','y1','x2','y2','target_box','base_logits','base_margin',
               'predicted_class','alternative_class','categories']
missing_3c = [k for k in required_3c if k not in globals()]
if missing_3c:
    raise RuntimeError('Run Steps 3A and 3B first; missing: ' + ', '.join(missing_3c))

# HIGH-ITERATION preset: 500 spatial controls per region (original: 30),
# seven strength levels (original: three), up to five candidate regions.
# All of these increase evaluation thoroughness/computation, NOT model-training epochs.
# For a short test you can set N_CONTROLS_3C=4 in a separate Colab code cell.
N_CONTROLS_3C = int(globals().get('N_CONTROLS_3C', 500))
BATCH_3C = int(globals().get('BATCH_3C', 2))  # small batches protect Colab memory
STRENGTHS_3C = (0.15, 0.30, 0.45, 0.60, 0.75, 0.90, 1.0)
METHODS_3C = ('blur','neutral_fill')
MAX_REGIONS_3C = int(globals().get('MAX_REGIONS_3C', 5))
SEED_3C = 2026
BOOTSTRAP_REPETITIONS_3C = 1500  # quantile precision diagnostic, NOT confidence of causal effects
if N_CONTROLS_3C < 3 or BATCH_3C < 1:
    raise ValueError('Use N_CONTROLS_3C >= 3 and BATCH_3C >= 1.')

model.eval()
for p in model.parameters(): p.requires_grad_(False)
source_3c = np.asarray(image.convert('RGB'), dtype=np.uint8).copy()
H3, W3 = source_3c.shape[:2]
ch3, cw3 = int(y2-y1), int(x2-x1)
crop_3c = source_3c[y1:y2,x1:x2].copy()
fg_3c = np.asarray(foreground, dtype=bool).copy()
if fg_3c.shape != (ch3,cw3):
    raise RuntimeError('Foreground and object box sizes do not match; rerun Steps 3A/3B.')

# Broccoli-only green color gate helps remove nearby meat/container; NOT a segmentation ground truth.
mask_note_3c = 'GrabCut foreground estimate from Step 3B'
target_name_3c = str(categories[int(predicted_class)]).lower()
if target_name_3c == 'broccoli':
    hsv = cv2.cvtColor(crop_3c, cv2.COLOR_RGB2HSV)
    green_seed = cv2.inRange(hsv, (33,35,20), (100,255,255)) > 0
    # Proximity gate admits dark florets near green pixels, and is still a heuristic.
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(23,23))
    green_near = cv2.dilate(green_seed.astype(np.uint8), k) > 0
    proposed = fg_3c & green_near
    if proposed.sum() >= 0.12 * fg_3c.sum():
        fg_3c = proposed
        mask_note_3c = 'GrabCut intersected with green-proximity HEURISTIC (broccoli only)'
    else:
        mask_note_3c += ' (green gate rejected as too small)'
print('[Step 3C]', mask_note_3c)
print('[Step 3C] Foreground coverage of box:', round(float(fg_3c.mean()),3))

# Keep selected original SLIC regions, but only where the refined foreground agrees.
selected_3c = []
minimum_pixels = max(40, int(0.015 * fg_3c.sum()))
for rec in record:
    rid = int(rec['region'])
    m = (segments == rid) & fg_3c
    if m.sum() >= minimum_pixels:
        selected_3c.append((rid,m,int(m.sum())))
    if len(selected_3c) >= MAX_REGIONS_3C: break
if not selected_3c:
    raise RuntimeError('No foreground candidate remains after refinement; inspect the mask.')
print('[Step 3C] Testing:', [(r,n) for r,_,n in selected_3c])

# Prepare content replacements once; no generative reconstruction or hallucinated features.
blur_3c = np.asarray(image.filter(ImageFilter.GaussianBlur(radius=14)).convert('RGB'), dtype=np.uint8)[y1:y2,x1:x2]
neutral_3c = np.median(crop_3c[fg_3c], axis=0).astype(np.uint8)
rng_3c = np.random.default_rng(SEED_3C)
fg_coords_3c = np.column_stack(np.where(fg_3c)).astype(np.int32)

# Controls: same number of pixels; compact nearest-foreground neighborhoods.
# IMPORTANT: same area != same shape / location / semantic evidence.
def control_mask_3c(n, original_mask):
    best = None
    best_overlap = 2.0
    for _ in range(30):
        yy,xx = fg_coords_3c[int(rng_3c.integers(len(fg_coords_3c)))].tolist()
        distances = ((fg_coords_3c[:,0].astype(np.float64)-yy)**2
                   + (fg_coords_3c[:,1].astype(np.float64)-xx)**2)
        idx = np.argpartition(distances, n-1)[:n]
        m = np.zeros((ch3,cw3), dtype=bool)
        m[fg_coords_3c[idx,0], fg_coords_3c[idx,1]] = True
        overlap = float((m & original_mask).sum()/n)
        if overlap < best_overlap:
            best,best_overlap=m,overlap
        if overlap <= 0.25: break
    return best,float(best_overlap)

# Inputs are generated ONE SMALL BATCH AT A TIME to avoid Colab crashes.
def edit_image_3c(mask, method, strength):
    edited = source_3c.copy()
    view = edited[y1:y2,x1:x2]
    old = crop_3c[mask].astype(np.float32)
    if method=='blur':
        target = blur_3c[mask].astype(np.float32)
    elif method=='neutral_fill':
        target = np.broadcast_to(neutral_3c, old.shape).astype(np.float32)
    else: raise ValueError(method)
    view[mask] = np.clip((1-strength)*old + strength*target,0,255).astype(np.uint8)
    return edited

@torch.inference_mode()
def fixed_roi_logits_batch_3c(images_np):
    ts = [to_tensor(PILImage.fromarray(a)).to(device) for a in images_np]
    transformed, _ = model.transform(ts, None)
    features = model.backbone(transformed.tensors)
    fixed_box = target_box.detach().to(device=device,dtype=torch.float32)
    roi_boxes = []
    for hh,ww in transformed.image_sizes:
        scale = fixed_box.new_tensor([ww/W3,hh/H3,ww/W3,hh/H3])
        roi_boxes.append((fixed_box*scale).unsqueeze(0))
    pooled = model.roi_heads.box_roi_pool(features,roi_boxes,transformed.image_sizes)
    embedding = model.roi_heads.box_head(pooled)
    logits, _ = model.roi_heads.box_predictor(embedding)
    return logits.detach().float().cpu().numpy()

# Detect any interface mismatch BEFORE paying for >100 inferences.
baseline_3c = fixed_roi_logits_batch_3c([source_3c])[0]
reference_3c = base_logits.detach().float().cpu().numpy()
max_disagreement_3c = float(np.max(np.abs(baseline_3c-reference_3c)))
print('[Step 3C] RoI logit consistency vs Step 3A:',round(max_disagreement_3c,6))
if max_disagreement_3c > 0.05:
    raise RuntimeError('RoI score mismatch with Step 3A; do NOT compare margins. Rerun Steps 1-3B.')
base_m_3c = float(baseline_3c[predicted_class]-baseline_3c[alternative_class])

jobs_3c=[]
control_overlap_3c={}
for rid,m,n in selected_3c:
    for mode in METHODS_3C:
        for s in STRENGTHS_3C:
            jobs_3c.append(dict(rid=rid,mask=m,mode=mode,strength=s,kind='observed',trial=-1))
    overlaps=[]
    for t in range(N_CONTROLS_3C):
        c,ov=control_mask_3c(n,m)
        overlaps.append(ov)
        for mode in METHODS_3C:
            jobs_3c.append(dict(rid=rid,mask=c,mode=mode,strength=1.0,kind='control',trial=t))
    control_overlap_3c[rid]=overlaps

# Resumable score cache. Exact job order is deterministic given the same data/config.
# Interrupted runs can be resumed in the SAME Colab file system / mounted Drive.
out_3c=Path(globals().get('C3R_OUTPUT_DIR', '/content/c3r_step1')); out_3c.mkdir(exist_ok=True,parents=True)
signature_3c = hashlib.sha256(
    source_3c.tobytes() + segments.astype(np.int32).tobytes() + fg_3c.tobytes()
    + reference_3c.astype(np.float32).tobytes()
    + json.dumps({'controls':N_CONTROLS_3C, 'strengths':STRENGTHS_3C,
                  'regions':[int(rid) for rid,_,_ in selected_3c],
                  'classes':[int(predicted_class), int(alternative_class)],
                  'seed':SEED_3C},sort_keys=True).encode('utf-8')
).hexdigest()[:16]
checkpoint_3c=out_3c/f'step3c_progress_{signature_3c}.jsonl'
finished_3c={}
if checkpoint_3c.exists():
    for line in checkpoint_3c.read_text(encoding='utf-8').splitlines():
        try:
            obj=json.loads(line)
            finished_3c[int(obj['job_id'])]=obj
        except (ValueError,KeyError,json.JSONDecodeError):
            pass
print('[Step 3C] Scoring',len(jobs_3c),'controlled image edits (batch=',BATCH_3C,');',
      len(finished_3c),'already cached.')
start_3c=time.time()
with checkpoint_3c.open('a', encoding='utf-8') as score_cache:
    for k in range(0,len(jobs_3c),BATCH_3C):
        portion=[(index,jobs_3c[index]) for index in range(k,min(k+BATCH_3C,len(jobs_3c)))
                 if index not in finished_3c]
        if portion:
            arrays=[edit_image_3c(j['mask'],j['mode'],j['strength']) for _,j in portion]
            logits_batch=fixed_roi_logits_batch_3c(arrays)
            for (index,j),logits in zip(portion,logits_batch):
                m_new=float(logits[predicted_class]-logits[alternative_class])
                result=dict(job_id=index,region=int(j['rid']),mode=j['mode'],strength=float(j['strength']),
                    kind=j['kind'],trial=int(j['trial']),margin=m_new,delta=base_m_3c-m_new)
                finished_3c[index]=result
                score_cache.write(json.dumps(result)+'\n')
            score_cache.flush()  # save completed batches for Colab disconnects
            del arrays,logits_batch
        if (k//BATCH_3C + 1)%25==0 or k+BATCH_3C>=len(jobs_3c):
            print(f' [Step 3C] {min(k+BATCH_3C,len(jobs_3c))}/{len(jobs_3c)} checked; {time.time()-start_3c:.0f}s')
        if device.type=='cuda' and (k//BATCH_3C)%60==0:
            torch.cuda.empty_cache()
records_3c=[finished_3c[i] for i in range(len(jobs_3c))]

# Conservative descriptive screening: two edits, seven strengths, 500 controls.
# The spatial controls may be correlated and are NOT independent statistical trials.
summary_3c=[]
bootstrap_rng_3c=np.random.default_rng(SEED_3C+93)
for rid,mask,n in selected_3c:
    bymode={}
    for mode in METHODS_3C:
        ob=sorted((r for r in records_3c if r['region']==rid and r['mode']==mode and r['kind']=='observed'),
                  key=lambda r:r['strength'])
        controls=np.array([r['delta'] for r in records_3c
            if r['region']==rid and r['mode']==mode and r['kind']=='control'],dtype=float)
        effect=float(ob[-1]['delta'])
        q95=float(np.quantile(np.abs(controls),.95))
        # Resampling the finite CONTROL POOL diagnoses how much its q95 estimate changes.
        # This is not a scientific CI for causality, robustness or a population effect.
        abs_controls=np.abs(controls)
        bq95=np.empty(BOOTSTRAP_REPETITIONS_3C,dtype=float)
        for bi in range(BOOTSTRAP_REPETITIONS_3C):
            sampled=bootstrap_rng_3c.choice(abs_controls,size=len(abs_controls),replace=True)
            bq95[bi]=np.quantile(sampled,.95)
        q95_range=[float(v) for v in np.quantile(bq95,[.025,.975])]
        stable_sign=bool(all(r['delta']*effect>0 for r in ob) and abs(effect)>0.05)
        exceed_fraction=float((1+np.count_nonzero(abs_controls>=abs(effect)))/(len(controls)+1))
        bymode[mode]=dict(full_effect=effect, doses=[dict(strength=r['strength'],delta=r['delta']) for r in ob],
            controls=controls.tolist(),q95_control_abs=q95,control_q95_bootstrap_interval=q95_range,
            descriptive_control_exceedance_fraction=exceed_fraction,
            outside_control_q95=bool(abs(effect)>q95),stable_sign=stable_sign)
    strong=(bymode['blur']['outside_control_q95'] and bymode['neutral_fill']['outside_control_q95']
            and bymode['blur']['stable_sign'] and bymode['neutral_fill']['stable_sign']
            and bymode['blur']['full_effect']*bymode['neutral_fill']['full_effect']>0
            and np.median(control_overlap_3c[rid]) <= 0.35)
    label=('provisional SUPPORT-LIKE region' if bymode['blur']['full_effect']>0
           else 'provisional CONFLICT-LIKE region') if strong else 'UNDETERMINED'
    if np.median(control_overlap_3c[rid]) > 0.35:
        print(' [Caution] Region',rid,': sampled controls substantially overlap the target region.')
    summary_3c.append(dict(region=rid,pixels=n,screening_label=label,
                           median_control_overlap=float(np.median(control_overlap_3c[rid])),methods=bymode))
    print(' Region',rid,'|',label,'| blur',round(bymode['blur']['full_effect'],3),
          '| fill',round(bymode['neutral_fill']['full_effect'],3),
          '| control q95',round(bymode['blur']['q95_control_abs'],3),round(bymode['neutral_fill']['q95_control_abs'],3))

# Six panel figure -- computed from real frozen detector scores, no invented numbers.
fig,axs=plt.subplots(2,3,figsize=(16,10),constrained_layout=True)
axs[0,0].imshow(source_3c)
axs[0,0].add_patch(Rectangle((x1,y1),cw3,ch3,fill=False,edgecolor='lime',lw=2))
axs[0,0].set_title('A. Clean / fixed target box')
axs[0,1].imshow(crop_3c)
axs[0,1].imshow(np.ma.masked_where(~fg_3c,fg_3c),cmap='Greens',alpha=.42)
axs[0,1].set_title('B. Refined foreground proxy (NOT ground truth)')
selected_colors=['red','deepskyblue','yellow','magenta','cyan']
axs[0,2].imshow(crop_3c)
for (rid,m,n),c in zip(selected_3c,selected_colors):
    axs[0,2].contour(m.astype(np.uint8),levels=[.5],colors=[c],linewidths=1.6)
    ys,xs=np.where(m)
    if len(xs): axs[0,2].text(float(np.median(xs)),float(np.median(ys)),f'R{rid}',color='white',fontsize=10,
                           bbox=dict(fc='black',alpha=.65,pad=1))
axs[0,2].set_title('C. Foreground-restricted candidates')
# real full-strength blur example for first region
rid0,m0,_=selected_3c[0]
axs[1,0].imshow(edit_image_3c(m0,'blur',1.0)[y1:y2,x1:x2])
axs[1,0].set_title(f'D. R{rid0}: actual blur intervention')
for mode,style in [('blur','-'),('neutral_fill','--')]:
    for s in summary_3c:
        vals=s['methods'][mode]['doses']
        axs[1,1].plot([v['strength'] for v in vals],[v['delta'] for v in vals],
                      marker='o',linestyle=style,label=f"R{s['region']} {mode}")
axs[1,1].axhline(0,color='black',lw=.8)
axs[1,1].set_xlabel('Intervention blend strength')
axs[1,1].set_ylabel('Δ logit margin (baseline − edited)')
axs[1,1].set_title('E. Dose-response on the same target RoI')
axs[1,1].legend(fontsize=7,ncol=2)
xs=[];ys=[];colors=[];names=[]
for s in summary_3c:
    for mode in METHODS_3C:
        names.append(f"R{s['region']}\n{mode.replace('neutral_fill','fill')}")
        xs.append(s['methods'][mode]['full_effect'])
        ys.append(s['methods'][mode]['q95_control_abs'])
        colors.append('tab:blue' if mode=='blur' else 'tab:orange')
idx=np.arange(len(xs))
axs[1,2].bar(idx,xs,color=colors,alpha=.85,label='Observed signed effect')
axs[1,2].scatter(idx,ys,marker='^',c='black',s=38,label='95th percentile |controls|')
axs[1,2].scatter(idx,-np.array(ys),marker='v',c='black',s=38)
axs[1,2].axhline(0,color='black',lw=.8)
axs[1,2].set_xticks(idx,names,fontsize=8)
axs[1,2].set_ylabel('Δ logit margin')
axs[1,2].set_title('F. Observed effects vs spatial controls')
axs[1,2].legend(fontsize=7)
for a in axs.flat:
    if a not in [axs[1,1],axs[1,2]]: a.axis('off')
fig.suptitle(f'C³R Step 3C | {categories[predicted_class]} vs {categories[alternative_class]} '
             f'| baseline logit margin {base_m_3c:+.3f}\n'
             'Controlled visual-region screening; semantic concepts, causality, and reliability NOT verified',fontsize=12)

fig_path_3c=out_3c/'step3c_region_screening.png'
json_path_3c=out_3c/'step3c_region_screening.json'
fig.savefig(fig_path_3c,dpi=180,bbox_inches='tight')
plt.show()
with open(json_path_3c,'w',encoding='utf-8') as f:
    json.dump(dict(image=str(globals().get('image_path','current image')),
        target_class=str(categories[predicted_class]),alternative_class=str(categories[alternative_class]),
        baseline_margin=base_m_3c,reference_consistency_error=max_disagreement_3c,
        foreground_mask_method=mask_note_3c,n_controls=N_CONTROLS_3C,
        regions=summary_3c,number_of_scored_edits=len(records_3c),strength_levels=STRENGTHS_3C,
        bootstrap_resamples=BOOTSTRAP_REPETITIONS_3C,semantic_concepts_verified=False,calibrated_reliability=None,
        cautions=['Color gate is a broccoli-specific heuristic, not segmentation ground truth',
                  '500 sampled spatial controls may be correlated, have repeated patterns and overlap target regions',
                  'Selection and testing share the same image; multiple candidate comparisons risk selection bias',
                  'Bootstrap interval measures finite-control quantile precision only; NOT statistical significance',
                  'Neutral fill and blur can be off-manifold',
                  'Object-specific semantic labels and cross-image validation remain pending']),f,indent=2)
print('\nSTEP 3C COMPLETED')
print('Figure:',fig_path_3c)
print('Data:',json_path_3c)
# Extra tidy CSV for statistical exploration and paper figures.
csv_path_3c=out_3c/'step3c_all_edits.csv'
with csv_path_3c.open('w',newline='',encoding='utf-8') as csv_file:
    writer=csv.DictWriter(csv_file,fieldnames=['job_id','region','mode','strength','kind','trial','margin','delta'])
    writer.writeheader()
    writer.writerows(records_3c)
print('Per-edit CSV:',csv_path_3c)
print('Resume checkpoint:',checkpoint_3c)
